In [2]:
# ============================================================
# NOTEBOOK 06
# DEPLOYMENT / INFERENCE PIPELINE
# ============================================================

import os
import joblib

import numpy as np
import pandas as pd

import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("✅ Deployment notebook imports loaded.")

TensorFlow Version: 2.21.0
✅ Deployment notebook imports loaded.


In [3]:
# ============================================================
# STEP 2
# PROJECT PATHS
# ============================================================

PROJECT_DIR = r"D:\IST\Aircraft_Engine_RUL"

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "final"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results"
)


print("Project:", PROJECT_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Final Models:", MODEL_DIR)

Project: D:\IST\Aircraft_Engine_RUL
Checkpoints: D:\IST\Aircraft_Engine_RUL\checkpoints
Final Models: D:\IST\Aircraft_Engine_RUL\models\final


In [4]:
# ============================================================
# STEP 3
# LOAD TRAIN-FITTED PREPROCESSING ARTIFACTS
# ============================================================

DL_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "phase_02_deep_learning_checkpoint.joblib"
)


dl_checkpoint = joblib.load(
    DL_CHECKPOINT_PATH
)


model_sensor_cols = dl_checkpoint[
    "model_sensor_cols"
]


fd002_op_scaler_model = dl_checkpoint[
    "fd002_op_scaler_model"
]

fd002_kmeans_model = dl_checkpoint[
    "fd002_kmeans_model"
]

fd002_model_condition_means = dl_checkpoint[
    "fd002_model_condition_means"
]

fd002_model_safe_stds = dl_checkpoint[
    "fd002_model_safe_stds"
]


fd003_model_means = dl_checkpoint[
    "fd003_model_means"
]

fd003_model_safe_stds = dl_checkpoint[
    "fd003_model_safe_stds"
]


raw_model_sensor_cols = [
    col.replace(
        "_model_norm",
        ""
    )
    for col in model_sensor_cols
]


print(
    "Sensors:",
    len(raw_model_sensor_cols)
)

print(
    raw_model_sensor_cols
)

print(
    "\n✅ Preprocessing artifacts loaded."
)

Sensors: 16
['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

✅ Preprocessing artifacts loaded.


In [5]:
# ============================================================
# STEP 4
# SERIALIZABLE CUSTOM LAYER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="CAUC"
)
class TemporalSum(
    tf.keras.layers.Layer
):

    def call(
        self,
        inputs
    ):

        return tf.reduce_sum(
            inputs,
            axis=1
        )


    def compute_output_shape(
        self,
        input_shape
    ):

        return (
            input_shape[0],
            input_shape[2]
        )


    def get_config(self):

        return super().get_config()


print(
    "✅ TemporalSum registered."
)

✅ TemporalSum registered.


In [6]:
# ============================================================
# STEP 5
# LOAD FINAL DEPLOYMENT MODELS
# ============================================================

FD002_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "CAUC_AeroRULNet_FD002.keras"
)

FD003_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "CAUC_AeroRULNet_FD003.keras"
)


print(
    "FD002 Model Exists:",
    os.path.exists(FD002_MODEL_PATH)
)

print(
    "FD003 Model Exists:",
    os.path.exists(FD003_MODEL_PATH)
)


final_model_fd002 = tf.keras.models.load_model(

    FD002_MODEL_PATH,

    custom_objects={
        "TemporalSum": TemporalSum
    },

    compile=False
)


final_model_fd003 = tf.keras.models.load_model(

    FD003_MODEL_PATH,

    custom_objects={
        "TemporalSum": TemporalSum
    },

    compile=False
)


print(
    "\nFD002 Parameters:",
    final_model_fd002.count_params()
)

print(
    "FD003 Parameters:",
    final_model_fd003.count_params()
)


assert (
    final_model_fd002.count_params()
    ==
    97378
)

assert (
    final_model_fd003.count_params()
    ==
    97378
)


print(
    "\n✅ Final deployment models loaded successfully."
)

FD002 Model Exists: True
FD003 Model Exists: True

FD002 Parameters: 97378
FD003 Parameters: 97378

✅ Final deployment models loaded successfully.


In [7]:
# ============================================================
# STEP 6
# DEPLOYMENT CONFIGURATION
# ============================================================

SEQUENCE_LENGTH = 50

RUL_CAP = 125


REQUIRED_OPERATIONAL_COLUMNS = [

    "operational_setting_1",
    "operational_setting_2",
    "operational_setting_3"
]


REQUIRED_SENSOR_COLUMNS = (
    raw_model_sensor_cols
)


print(
    "Sequence Length:",
    SEQUENCE_LENGTH
)

print(
    "Number of Sensors:",
    len(REQUIRED_SENSOR_COLUMNS)
)

Sequence Length: 50
Number of Sensors: 16


In [8]:
# ============================================================
# STEP 7
# MAINTENANCE DECISION SUPPORT
# ============================================================

def classify_maintenance_risk(
    rul
):

    if rul <= 30:

        return "Critical"

    elif rul <= 60:

        return "High Risk"

    elif rul <= 90:

        return "Moderate"

    else:

        return "Healthy"



def maintenance_recommendation(
    risk
):

    recommendations = {

        "Critical":
            "Immediate maintenance required",

        "High Risk":
            "Schedule maintenance soon",

        "Moderate":
            "Increase monitoring frequency",

        "Healthy":
            "Continue routine monitoring"
    }

    return recommendations[
        risk
    ]



def maintenance_priority(
    risk
):

    priorities = {

        "Critical": "P1",

        "High Risk": "P2",

        "Moderate": "P3",

        "Healthy": "P4"
    }

    return priorities[
        risk
    ]


print(
    "✅ Maintenance decision logic ready."
)

✅ Maintenance decision logic ready.


In [9]:
# ============================================================
# STEP 8
# VALIDATE NEW ENGINE DATA
# ============================================================

def validate_engine_input(
    engine_df,
    dataset_type
):

    if not isinstance(
        engine_df,
        pd.DataFrame
    ):

        raise TypeError(
            "Input must be a Pandas DataFrame."
        )


    dataset_type = (
        dataset_type
        .upper()
        .strip()
    )


    if dataset_type not in [
        "FD002",
        "FD003"
    ]:

        raise ValueError(
            "dataset_type must be FD002 or FD003."
        )


    required_cols = [

        "cycle",

        *REQUIRED_SENSOR_COLUMNS
    ]


    # FD002 requires operational settings
    if dataset_type == "FD002":

        required_cols += (
            REQUIRED_OPERATIONAL_COLUMNS
        )


    missing_columns = [

        col
        for col in required_cols
        if col not in engine_df.columns
    ]


    if missing_columns:

        raise ValueError(

            f"Missing required columns: "
            f"{missing_columns}"
        )


    # Numeric NaN check
    if engine_df[
        required_cols
    ].isna().any().any():

        raise ValueError(
            "Input contains missing values."
        )


    # Infinite-value check
    if np.isinf(
        engine_df[
            required_cols
        ].to_numpy(
            dtype=float
        )
    ).any():

        raise ValueError(
            "Input contains infinite values."
        )


    if len(engine_df) == 0:

        raise ValueError(
            "Engine data is empty."
        )


    print(
        f"✅ {dataset_type} engine input validated."
    )

    print(
        "Observed Cycles:",
        len(engine_df)
    )


    return True

In [10]:
# ============================================================
# STEP 9
# FD002 DEPLOYMENT PREPROCESSING
# ============================================================

def preprocess_fd002_engine(
    engine_df
):

    df = (
        engine_df
        .copy()
        .sort_values("cycle")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Scale operational conditions
    # --------------------------------------------------------

    op_scaled = (
        fd002_op_scaler_model.transform(

            df[
                REQUIRED_OPERATIONAL_COLUMNS
            ]
        )
    )


    # --------------------------------------------------------
    # Assign train-learned operating condition
    # --------------------------------------------------------

    df[
        "operating_condition_model"
    ] = fd002_kmeans_model.predict(
        op_scaled
    )


    # --------------------------------------------------------
    # Normalize each sensor using the corresponding
    # train-derived operating-condition statistics
    # --------------------------------------------------------

    for raw_sensor, norm_sensor in zip(

        raw_model_sensor_cols,

        model_sensor_cols

    ):

        df[
            norm_sensor
        ] = np.nan


        for condition in (
            fd002_model_condition_means.index
        ):

            mask = (

                df[
                    "operating_condition_model"
                ]

                ==

                condition
            )


            mean_value = (
                fd002_model_condition_means
                .loc[
                    condition,
                    raw_sensor
                ]
            )


            std_value = (
                fd002_model_safe_stds
                .loc[
                    condition,
                    raw_sensor
                ]
            )


            df.loc[
                mask,
                norm_sensor
            ] = (

                df.loc[
                    mask,
                    raw_sensor
                ]

                -

                mean_value

            ) / std_value


    return df

In [11]:
# ============================================================
# STEP 10
# FD003 DEPLOYMENT PREPROCESSING
# ============================================================

def preprocess_fd003_engine(
    engine_df
):

    df = (
        engine_df
        .copy()
        .sort_values("cycle")
        .reset_index(drop=True)
    )


    for raw_sensor, norm_sensor in zip(

        raw_model_sensor_cols,

        model_sensor_cols

    ):

        mean_value = (
            fd003_model_means[
                raw_sensor
            ]
        )

        std_value = (
            fd003_model_safe_stds[
                raw_sensor
            ]
        )


        df[
            norm_sensor
        ] = (

            df[
                raw_sensor
            ]

            -

            mean_value

        ) / std_value


    return df

In [12]:
# ============================================================
# STEP 11
# CREATE MODEL INPUT SEQUENCE
# ============================================================

def build_deployment_sequence(
    processed_df,
    sequence_length=50
):

    sensor_data = (

        processed_df[
            model_sensor_cols
        ]

        .to_numpy(
            dtype=np.float32
        )
    )


    observed_cycles = len(
        sensor_data
    )


    # --------------------------------------------------------
    # Enough history
    # --------------------------------------------------------

    if observed_cycles >= sequence_length:

        sequence = sensor_data[
            -sequence_length:
        ]

        padding_count = 0


    # --------------------------------------------------------
    # Short history
    # --------------------------------------------------------

    else:

        padding_count = (
            sequence_length
            -
            observed_cycles
        )


        first_state = (
            sensor_data[
                0:1
            ]
        )


        padding = np.repeat(

            first_state,

            repeats=
                padding_count,

            axis=0
        )


        sequence = np.concatenate(

            [
                padding,
                sensor_data
            ],

            axis=0
        )


    sequence = np.expand_dims(
        sequence,
        axis=0
    )


    assert sequence.shape == (
        1,
        sequence_length,
        16
    )


    return (
        sequence,
        padding_count
    )

In [13]:
# ============================================================
# STEP 12
# COMPLETE DEPLOYMENT INFERENCE PIPELINE
# ============================================================

def predict_engine_rul(
    engine_df,
    dataset_type
):

    dataset_type = (
        dataset_type
        .upper()
        .strip()
    )


    # ========================================================
    # 1. Validate
    # ========================================================

    validate_engine_input(
        engine_df,
        dataset_type
    )


    # ========================================================
    # 2. Preprocess
    # ========================================================

    if dataset_type == "FD002":

        processed_df = (
            preprocess_fd002_engine(
                engine_df
            )
        )

        model = (
            final_model_fd002
        )


    else:

        processed_df = (
            preprocess_fd003_engine(
                engine_df
            )
        )

        model = (
            final_model_fd003
        )


    # ========================================================
    # 3. Quality check
    # ========================================================

    model_values = (
        processed_df[
            model_sensor_cols
        ].to_numpy()
    )


    if not np.isfinite(
        model_values
    ).all():

        raise ValueError(
            "Preprocessed data contains NaN/Inf."
        )


    # ========================================================
    # 4. Build sequence
    # ========================================================

    sequence, padding_count = (
        build_deployment_sequence(
            processed_df,
            SEQUENCE_LENGTH
        )
    )


    # ========================================================
    # 5. Predict RUL
    # ========================================================

    predicted_rul = float(

        model.predict(
            sequence,
            verbose=0
        )
        .reshape(-1)[0]
    )


    # Avoid physically meaningless negative RUL
    predicted_rul = max(
        predicted_rul,
        0.0
    )


    # ========================================================
    # 6. Maintenance decision
    # ========================================================

    risk = classify_maintenance_risk(
        predicted_rul
    )


    priority = maintenance_priority(
        risk
    )


    recommendation = (
        maintenance_recommendation(
            risk
        )
    )


    # ========================================================
    # 7. Result
    # ========================================================

    result = {

        "Dataset":
            dataset_type,

        "Observed_Cycles":
            len(engine_df),

        "Sequence_Length":
            SEQUENCE_LENGTH,

        "Padding_Used":
            padding_count,

        "Predicted_RUL":
            round(
                predicted_rul,
                2
            ),

        "Risk_Level":
            risk,

        "Maintenance_Priority":
            priority,

        "Recommendation":
            recommendation
    }


    return result

In [14]:
# ============================================================
# STEP 12A
# LOAD NASA TEST DATA FOR PIPELINE VERIFICATION
# ============================================================

DATA_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw"
)


FD002_TEST_PATH = os.path.join(
    DATA_DIR,
    "test_FD002.txt"
)

FD003_TEST_PATH = os.path.join(
    DATA_DIR,
    "test_FD003.txt"
)


column_names = (

    [
        "engine_id",
        "cycle"
    ]

    +

    [
        "operational_setting_1",
        "operational_setting_2",
        "operational_setting_3"
    ]

    +

    [
        f"sensor_{i}"
        for i in range(1, 22)
    ]
)


test_fd002_raw = pd.read_csv(
    FD002_TEST_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)


test_fd003_raw = pd.read_csv(
    FD003_TEST_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)


print("=" * 65)
print("DEPLOYMENT VERIFICATION DATA LOADED")
print("=" * 65)

print(
    "FD002 Shape:",
    test_fd002_raw.shape
)

print(
    "FD002 Engines:",
    test_fd002_raw["engine_id"].nunique()
)

print(
    "\nFD003 Shape:",
    test_fd003_raw.shape
)

print(
    "FD003 Engines:",
    test_fd003_raw["engine_id"].nunique()
)

DEPLOYMENT VERIFICATION DATA LOADED
FD002 Shape: (33991, 26)
FD002 Engines: 259

FD003 Shape: (16596, 26)
FD003 Engines: 100


In [15]:
# ============================================================
# STEP 13
# TEST COMPLETE PIPELINE - FD002
# ============================================================

sample_engine_fd002 = (

    test_fd002_raw[
        test_fd002_raw[
            "engine_id"
        ] == 1
    ]

    .copy()
)


fd002_sample_result = (
    predict_engine_rul(

        sample_engine_fd002,

        dataset_type="FD002"
    )
)


print("=" * 70)
print("FD002 SAMPLE DEPLOYMENT RESULT")
print("=" * 70)


for key, value in (
    fd002_sample_result.items()
):

    print(
        f"{key}: {value}"
    )

✅ FD002 engine input validated.
Observed Cycles: 258
FD002 SAMPLE DEPLOYMENT RESULT
Dataset: FD002
Observed_Cycles: 258
Sequence_Length: 50
Padding_Used: 0
Predicted_RUL: 14.86
Risk_Level: Critical
Maintenance_Priority: P1
Recommendation: Immediate maintenance required


In [16]:
# ============================================================
# STEP 14
# TEST COMPLETE PIPELINE - FD003
# ============================================================

sample_engine_fd003 = (
    test_fd003_raw[
        test_fd003_raw["engine_id"] == 1
    ]
    .copy()
)


fd003_sample_result = predict_engine_rul(
    sample_engine_fd003,
    dataset_type="FD003"
)


print("=" * 70)
print("FD003 SAMPLE DEPLOYMENT RESULT")
print("=" * 70)

for key, value in fd003_sample_result.items():
    print(
        f"{key}: {value}"
    )

✅ FD003 engine input validated.
Observed Cycles: 233
FD003 SAMPLE DEPLOYMENT RESULT
Dataset: FD003
Observed_Cycles: 233
Sequence_Length: 50
Padding_Used: 0
Predicted_RUL: 41.54
Risk_Level: High Risk
Maintenance_Priority: P2
Recommendation: Schedule maintenance soon


In [17]:
# ============================================================
# STEP 15
# SAVE DEPLOYMENT CONFIGURATION
# ============================================================

DEPLOYMENT_DIR = os.path.join(
    PROJECT_DIR,
    "deployment"
)

os.makedirs(
    DEPLOYMENT_DIR,
    exist_ok=True
)


DEPLOYMENT_CONFIG = {

    "sequence_length": 50,

    "rul_cap": 125,

    "supported_datasets": [
        "FD002",
        "FD003"
    ],

    "model_sensor_cols":
        model_sensor_cols,

    "raw_sensor_cols":
        raw_model_sensor_cols,

    "operational_columns":
        REQUIRED_OPERATIONAL_COLUMNS,

    "risk_thresholds": {
        "Critical": "<=30",
        "High Risk": "31-60",
        "Moderate": "61-90",
        "Healthy": ">90"
    },

    "priority_mapping": {
        "Critical": "P1",
        "High Risk": "P2",
        "Moderate": "P3",
        "Healthy": "P4"
    }
}


DEPLOYMENT_CONFIG_PATH = os.path.join(
    DEPLOYMENT_DIR,
    "deployment_config.pkl"
)


joblib.dump(
    DEPLOYMENT_CONFIG,
    DEPLOYMENT_CONFIG_PATH
)


print(
    "✅ Deployment configuration saved:"
)

print(
    DEPLOYMENT_CONFIG_PATH
)

✅ Deployment configuration saved:
D:\IST\Aircraft_Engine_RUL\deployment\deployment_config.pkl


In [18]:
# ============================================================
# STEP 16
# CSV INPUT LOADER
# ============================================================

def load_engine_csv(
    file_path
):

    if not os.path.exists(
        file_path
    ):

        raise FileNotFoundError(
            f"File not found: {file_path}"
        )


    df = pd.read_csv(
        file_path
    )


    if df.empty:

        raise ValueError(
            "Uploaded CSV is empty."
        )


    print(
        "✅ CSV loaded successfully."
    )

    print(
        "Shape:",
        df.shape
    )


    return df

In [19]:
# ============================================================
# STEP 17
# MULTIPLE-ENGINE PREDICTION PIPELINE
# ============================================================

def predict_multiple_engines(
    df,
    dataset_type
):

    results = []


    # --------------------------------------------------------
    # If engine_id exists, predict separately for each engine
    # --------------------------------------------------------

    if "engine_id" in df.columns:

        grouped_data = df.groupby(
            "engine_id",
            sort=True
        )

    else:

        # Treat entire file as one engine
        grouped_data = [
            (1, df)
        ]


    for engine_id, engine_df in grouped_data:

        try:

            result = predict_engine_rul(
                engine_df,
                dataset_type
            )

            result[
                "Engine_ID"
            ] = engine_id

            result[
                "Status"
            ] = "Success"

            result[
                "Error"
            ] = None


        except Exception as e:

            result = {

                "Engine_ID":
                    engine_id,

                "Dataset":
                    dataset_type,

                "Status":
                    "Failed",

                "Error":
                    str(e)
            }


        results.append(
            result
        )


    results_df = pd.DataFrame(
        results
    )


    return results_df

In [20]:
# ============================================================
# STEP 18
# TEST MULTIPLE ENGINE PREDICTION
# ============================================================

sample_fd002_multi = (
    test_fd002_raw[
        test_fd002_raw[
            "engine_id"
        ].isin(
            [1, 2, 3, 4, 5]
        )
    ]
    .copy()
)


multi_fd002_results = (
    predict_multiple_engines(

        sample_fd002_multi,

        dataset_type="FD002"
    )
)


display(
    multi_fd002_results[
        [
            "Engine_ID",
            "Observed_Cycles",
            "Predicted_RUL",
            "Risk_Level",
            "Maintenance_Priority",
            "Recommendation",
            "Status"
        ]
    ]
)

✅ FD002 engine input validated.
Observed Cycles: 258
✅ FD002 engine input validated.
Observed Cycles: 55
✅ FD002 engine input validated.
Observed Cycles: 165
✅ FD002 engine input validated.
Observed Cycles: 86
✅ FD002 engine input validated.
Observed Cycles: 148


,Engine_ID,Observed_Cycles,Predicted_RUL,Risk_Level,Maintenance_Priority,Recommendation,Status
0,1,258,14.86,Critical,P1,Immediate maintenance required,Success
1,2,55,103.96,Healthy,P4,Continue routine monitoring,Success
2,3,165,102.08,Healthy,P4,Continue routine monitoring,Success
3,4,86,75.37,Moderate,P3,Increase monitoring frequency,Success
4,5,148,12.32,Critical,P1,Immediate maintenance required,Success


In [21]:
# ============================================================
# STEP 19
# MAINTENANCE PRIORITY SORTING
# ============================================================

PRIORITY_ORDER = {
    "P1": 1,
    "P2": 2,
    "P3": 3,
    "P4": 4
}


def sort_by_maintenance_priority(
    results_df
):

    df = (
        results_df
        .copy()
    )


    if "Maintenance_Priority" not in df.columns:

        return df


    df[
        "Priority_Order"
    ] = (

        df[
            "Maintenance_Priority"
        ]
        .map(
            PRIORITY_ORDER
        )
    )


    df = (
        df
        .sort_values(
            [
                "Priority_Order",
                "Predicted_RUL"
            ],
            ascending=[
                True,
                True
            ]
        )
        .reset_index(drop=True)
    )


    return df

In [22]:
# ============================================================
# TEST PRIORITY SORTING
# ============================================================

sorted_fd002_results = (
    sort_by_maintenance_priority(
        multi_fd002_results
    )
)


display(
    sorted_fd002_results[
        [
            "Engine_ID",
            "Predicted_RUL",
            "Risk_Level",
            "Maintenance_Priority",
            "Recommendation"
        ]
    ]
)

,Engine_ID,Predicted_RUL,Risk_Level,Maintenance_Priority,Recommendation
0,5,12.32,Critical,P1,Immediate maintenance required
1,1,14.86,Critical,P1,Immediate maintenance required
2,4,75.37,Moderate,P3,Increase monitoring frequency
3,3,102.08,Healthy,P4,Continue routine monitoring
4,2,103.96,Healthy,P4,Continue routine monitoring


In [23]:
# ============================================================
# STEP 20
# FLEET MAINTENANCE SUMMARY
# ============================================================

def generate_fleet_summary(
    results_df
):

    successful = results_df[
        results_df[
            "Status"
        ] == "Success"
    ]


    summary = {

        "Total_Engines":
            len(results_df),

        "Successful_Predictions":
            len(successful),

        "Failed_Predictions":
            (
                results_df[
                    "Status"
                ] == "Failed"
            ).sum(),

        "Critical_Engines":
            (
                successful[
                    "Risk_Level"
                ] == "Critical"
            ).sum(),

        "High_Risk_Engines":
            (
                successful[
                    "Risk_Level"
                ] == "High Risk"
            ).sum(),

        "Moderate_Engines":
            (
                successful[
                    "Risk_Level"
                ] == "Moderate"
            ).sum(),

        "Healthy_Engines":
            (
                successful[
                    "Risk_Level"
                ] == "Healthy"
            ).sum()
    }


    if len(successful) > 0:

        summary[
            "Average_Predicted_RUL"
        ] = successful[
            "Predicted_RUL"
        ].mean()

        summary[
            "Minimum_Predicted_RUL"
        ] = successful[
            "Predicted_RUL"
        ].min()

    else:

        summary[
            "Average_Predicted_RUL"
        ] = np.nan

        summary[
            "Minimum_Predicted_RUL"
        ] = np.nan


    return summary

In [24]:
# ============================================================
# TEST FLEET SUMMARY
# ============================================================

fleet_summary = generate_fleet_summary(
    multi_fd002_results
)


print("=" * 65)
print("FLEET MAINTENANCE SUMMARY")
print("=" * 65)

for key, value in fleet_summary.items():

    print(
        f"{key}: {value}"
    )

FLEET MAINTENANCE SUMMARY
Total_Engines: 5
Successful_Predictions: 5
Failed_Predictions: 0
Critical_Engines: 2
High_Risk_Engines: 0
Moderate_Engines: 1
Healthy_Engines: 2
Average_Predicted_RUL: 61.717999999999996
Minimum_Predicted_RUL: 12.32


In [25]:
# ============================================================
# STEP 21
# SAVE DEPLOYMENT PREDICTION RESULTS
# ============================================================

DEPLOYMENT_OUTPUT_DIR = os.path.join(
    DEPLOYMENT_DIR,
    "outputs"
)

os.makedirs(
    DEPLOYMENT_OUTPUT_DIR,
    exist_ok=True
)


SAMPLE_OUTPUT_PATH = os.path.join(
    DEPLOYMENT_OUTPUT_DIR,
    "sample_fd002_predictions.csv"
)


sorted_fd002_results.to_csv(
    SAMPLE_OUTPUT_PATH,
    index=False
)


print(
    "✅ Sample deployment predictions saved:"
)

print(
    SAMPLE_OUTPUT_PATH
)

✅ Sample deployment predictions saved:
D:\IST\Aircraft_Engine_RUL\deployment\outputs\sample_fd002_predictions.csv


In [26]:
# ============================================================
# STEP 22
# FINAL DEPLOYMENT PIPELINE HEALTH CHECK
# ============================================================

checks = {

    "FD002 model loaded":
        final_model_fd002 is not None,

    "FD003 model loaded":
        final_model_fd003 is not None,

    "FD002 parameters correct":
        final_model_fd002.count_params()
        == 97378,

    "FD003 parameters correct":
        final_model_fd003.count_params()
        == 97378,

    "16 sensors loaded":
        len(
            raw_model_sensor_cols
        ) == 16,

    "Deployment config saved":
        os.path.exists(
            DEPLOYMENT_CONFIG_PATH
        ),

    "FD002 sample prediction works":
        fd002_sample_result[
            "Predicted_RUL"
        ] >= 0,

    "FD003 sample prediction works":
        fd003_sample_result[
            "Predicted_RUL"
        ] >= 0
}


print("=" * 70)
print("DEPLOYMENT PIPELINE HEALTH CHECK")
print("=" * 70)


all_checks_passed = True


for check_name, status in checks.items():

    print(
        f"{'✅' if status else '❌'} "
        f"{check_name}"
    )

    if not status:
        all_checks_passed = False


if all_checks_passed:

    print(
        "\n🎉 DEPLOYMENT PIPELINE READY."
    )

else:

    print(
        "\n❌ Some deployment checks failed."
    )

DEPLOYMENT PIPELINE HEALTH CHECK
✅ FD002 model loaded
✅ FD003 model loaded
✅ FD002 parameters correct
✅ FD003 parameters correct
✅ 16 sensors loaded
✅ Deployment config saved
✅ FD002 sample prediction works
✅ FD003 sample prediction works

🎉 DEPLOYMENT PIPELINE READY.


In [27]:
# ============================================================
# FIGURE 7(a)
# ACTUAL VS PREDICTED RUL - FD002
# CAUC-AeroRULNet
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# IMPORTANT:
# These should be the SAME final validation arrays used for
# the reported CAUC metrics:
#
# MAE  = 10.469964
# RMSE = 13.462717
# R2   = 0.891859
# ------------------------------------------------------------

y_true_fd002 = np.asarray(
    y_val_fd002_recreated
).reshape(-1)

y_pred_fd002 = np.asarray(
    pred_val_fd002_cauc
).reshape(-1)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(y_true_fd002) == len(y_pred_fd002)

print("FD002 Samples:", len(y_true_fd002))


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(7, 6))

plt.scatter(
    y_true_fd002,
    y_pred_fd002,
    alpha=0.35,
    s=18
)

min_val = min(
    y_true_fd002.min(),
    y_pred_fd002.min()
)

max_val = max(
    y_true_fd002.max(),
    y_pred_fd002.max()
)

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    linewidth=2,
    label="Ideal Prediction"
)

plt.xlabel("Actual RUL (Cycles)")
plt.ylabel("Predicted RUL (Cycles)")

plt.title(
    "CAUC-AeroRULNet: Actual vs Predicted RUL - FD002"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()

NameError: name 'y_val_fd002_recreated' is not defined